# 96 — Build a unified nodal stacked-gather catalog (SAFE, v2)

This notebook combines the two completed nodal stack branches into one **read-only integration catalog**:

1. **Geode-linked nodal stacks** produced by notebook 95 and stored in SQLite:
   - `nodal_stacks`
   - `nodal_stack_members`
   - `nodal_stack_files`
   - `nodal_stack_processing_errors`

2. **Nodal-only stacks** produced by the nodal-only stacking notebook and stored as CSV exports:
   - `nodal_only_stacks.csv`
   - `nodal_only_stack_members.csv`
   - `nodal_only_stack_files.csv`
   - `nodal_only_stack_trace_contributions.csv`
   - `nodal_only_stack_processing_errors.csv`

The notebook:

- preserves the provenance distinction between Geode-linked and nodal-only source groups;
- standardizes source-position, component, path, status, and QC fields;
- verifies that indexed waveform products exist;
- creates candidate source-position clusters using a configurable **0.25 m tolerance**;
- identifies clusters containing both Geode-linked and nodal-only products;
- writes unified stack, file, member, receiver, and error catalogs;
- does **not** alter the original SQLite database or either upstream output tree.

### Important interpretation

A source-position cluster is a **candidate proximity grouping**, not proof that the records are repeated blows at the same source. Waveform correlation and acquisition context must be checked before any stacks are merged.


**v2 fix:** replaces the fragile pandas `apply()` source-cluster metadata construction that caused `KeyError: canonical_source_x_m`.

## 1. Configuration

In [1]:
from pathlib import Path
import sqlite3
import json
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')

# Notebook-95 Geode-linked products.
CATALOG_DB = PROJECT_ROOT / 'catalog' / 'lbssp_shot_catalog.sqlite'

# Nodal-only stack products.
NODAL_ONLY_ROOT = PROJECT_ROOT / 'nodal_only_stacked_shot_gathers'
NODAL_ONLY_EXPORT_ROOT = NODAL_ONLY_ROOT / 'catalog_exports'

# Unified outputs owned only by this notebook.
OUT_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_catalog'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_POSITION_TOLERANCE_M = 0.25

# Set to a list such as ['T1'] to restrict output, or None for all lines.
TARGET_LINES = None

# When True, absence of either input branch is fatal.
REQUIRE_BOTH_BRANCHES = True

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

print('SQLite input:', CATALOG_DB)
print('Nodal-only CSV input:', NODAL_ONLY_EXPORT_ROOT)
print('Unified output:', OUT_ROOT)
print('Shot-position tolerance:', SHOT_POSITION_TOLERANCE_M, 'm')
print('This notebook does not modify SQLite.')


SQLite input: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
Nodal-only CSV input: /Volumes/tachyon/LBSSP_DATA/nodal_only_stacked_shot_gathers/catalog_exports
Unified output: /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog
Shot-position tolerance: 0.25 m
This notebook does not modify SQLite.


## 2. Helpers

In [2]:
def as_bool(series, default=False):
    if series is None:
        return pd.Series(dtype=bool)
    return (
        series.fillna(default)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['true', '1', 'yes', 'y'])
    )


def numeric(frame, columns):
    out = frame.copy()
    for column in columns:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors='coerce')
    return out


def text_or_none(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def existing_path(value):
    path_text = text_or_none(value)
    return bool(path_text) and Path(path_text).exists()


def normalize_component(value):
    text = str(value).strip().upper()
    if text.endswith('Z'):
        return 'Z'
    if text.endswith('N'):
        return 'N'
    if text.endswith('E'):
        return 'E'
    return text


def first_present(frame, candidates, default=np.nan):
    for column in candidates:
        if column in frame.columns:
            return frame[column]
    return pd.Series(default, index=frame.index)


def cluster_positions(frame, *, tolerance_m):
    """Assign line-specific source-position proximity clusters.

    This implementation deliberately builds the cluster metadata as explicit
    records rather than relying on ``DataFrame.apply``. That avoids a pandas
    edge case in which a Series-returning apply can yield no named columns.
    """
    metadata_columns = [
        'source_cluster_number',
        'canonical_source_x_m',
        'cluster_min_x_m',
        'cluster_max_x_m',
        'cluster_span_m',
        'cluster_n_stacks',
        'cluster_contains_geode_linked',
        'cluster_contains_nodal_only',
    ]

    if frame.empty:
        out = frame.copy()
        for column in metadata_columns:
            if column not in out.columns:
                out[column] = pd.Series(dtype=float if column.endswith('_m') else object)
        out['source_cluster_id'] = pd.Series(dtype=object)
        return out

    output_parts = []

    for line, group in frame.groupby('line', dropna=False, sort=True):
        group = group.sort_values(
            ['source_x_m', 'source_position_authority', 'stack_id'],
            na_position='last',
        ).copy()

        assignments = {}
        current_members = []
        cluster_number = 0

        for idx, x_value in group['source_x_m'].items():
            if pd.isna(x_value):
                continue

            x = float(x_value)
            if not current_members:
                cluster_number += 1
                current_members = [idx]
                assignments[idx] = cluster_number
                continue

            current_x = float(np.nanmedian(group.loc[current_members, 'source_x_m']))
            if abs(x - current_x) <= float(tolerance_m):
                current_members.append(idx)
                assignments[idx] = cluster_number
            else:
                cluster_number += 1
                current_members = [idx]
                assignments[idx] = cluster_number

        group['source_cluster_number'] = pd.Series(assignments, dtype='float64')

        metadata_records = []
        for cluster in sorted(
            int(value)
            for value in group['source_cluster_number'].dropna().unique()
        ):
            members = group.loc[group.source_cluster_number.eq(cluster)].copy()
            truth = members.loc[
                members.source_position_authority.eq('surveyed_geode_source'),
                'source_x_m',
            ].dropna()

            canonical = (
                float(np.nanmedian(truth.to_numpy(dtype=float)))
                if len(truth)
                else float(np.nanmedian(members.source_x_m.to_numpy(dtype=float)))
            )
            xmin = float(members.source_x_m.min())
            xmax = float(members.source_x_m.max())

            metadata_records.append({
                'source_cluster_number': float(cluster),
                'canonical_source_x_m': canonical,
                'cluster_min_x_m': xmin,
                'cluster_max_x_m': xmax,
                'cluster_span_m': xmax - xmin,
                'cluster_n_stacks': int(len(members)),
                'cluster_contains_geode_linked': bool(
                    members.catalog_branch.eq('geode_linked').any()
                ),
                'cluster_contains_nodal_only': bool(
                    members.catalog_branch.eq('nodal_only').any()
                ),
            })

        metadata = pd.DataFrame(metadata_records, columns=metadata_columns)
        if len(metadata):
            group = group.merge(
                metadata,
                on='source_cluster_number',
                how='left',
                validate='many_to_one',
            )
        else:
            for column in metadata_columns[1:]:
                group[column] = np.nan
            group['cluster_contains_geode_linked'] = False
            group['cluster_contains_nodal_only'] = False

        line_token = 'UNKNOWN' if pd.isna(line) else re.sub(
            r'[^A-Za-z0-9_.-]+', '_', str(line)
        )
        group['source_cluster_id'] = group.source_cluster_number.map(
            lambda value: (
                f'{line_token}_SRC_{int(value):04d}'
                if pd.notna(value) else None
            )
        )
        output_parts.append(group)

    out = pd.concat(output_parts, ignore_index=True, sort=False)

    # Guarantee the schema expected by downstream cells even when all source
    # positions are missing for a line.
    for column in metadata_columns[1:]:
        if column not in out.columns:
            out[column] = np.nan
    if 'source_cluster_id' not in out.columns:
        out['source_cluster_id'] = None

    return out


## 3. Load Geode-linked nodal stacks from notebook 95

In [3]:
GEODE_REQUIRED_TABLES = [
    'nodal_stacks',
    'nodal_stack_members',
    'nodal_stack_files',
    'nodal_stack_processing_errors',
]

if not CATALOG_DB.exists():
    if REQUIRE_BOTH_BRANCHES:
        raise FileNotFoundError(f'Missing SQLite catalog: {CATALOG_DB}')
    geode_stacks_raw = pd.DataFrame()
    geode_members_raw = pd.DataFrame()
    geode_files_raw = pd.DataFrame()
    geode_errors_raw = pd.DataFrame()
else:
    uri = f'file:{CATALOG_DB}?mode=ro'
    with sqlite3.connect(uri, uri=True) as connection:
        available_tables = set(
            pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table'",
                connection,
            )['name']
        )
        missing = sorted(set(GEODE_REQUIRED_TABLES) - available_tables)
        if missing:
            raise RuntimeError(
                f'SQLite catalog is missing notebook-95 tables: {missing}'
            )

        geode_stacks_raw = pd.read_sql(
            'SELECT * FROM nodal_stacks', connection
        )
        geode_members_raw = pd.read_sql(
            'SELECT * FROM nodal_stack_members', connection
        )
        geode_files_raw = pd.read_sql(
            'SELECT * FROM nodal_stack_files', connection
        )
        geode_errors_raw = pd.read_sql(
            'SELECT * FROM nodal_stack_processing_errors', connection
        )

print('Geode-linked stacks:', len(geode_stacks_raw))
print('Geode-linked members:', len(geode_members_raw))
print('Geode-linked indexed files:', len(geode_files_raw))
print('Geode-linked processing errors:', len(geode_errors_raw))


Geode-linked stacks: 1
Geode-linked members: 6
Geode-linked indexed files: 3
Geode-linked processing errors: 193


## 4. Load nodal-only stack CSV exports

In [4]:
NODAL_ONLY_FILES = {
    'stacks': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stacks.csv',
    'members': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_members.csv',
    'files': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_files.csv',
    'receivers': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_trace_contributions.csv',
    'errors': NODAL_ONLY_EXPORT_ROOT / 'nodal_only_stack_processing_errors.csv',
}

missing_nodal_only = [
    name for name, path in NODAL_ONLY_FILES.items() if not path.exists()
]

if missing_nodal_only and REQUIRE_BOTH_BRANCHES:
    details = '\n'.join(
        f'  {name}: {NODAL_ONLY_FILES[name]}'
        for name in missing_nodal_only
    )
    raise FileNotFoundError(
        'Missing nodal-only catalog exports. Run the nodal-only stacking '
        f'notebook first.\n{details}'
    )

def read_optional_csv(path):
    return pd.read_csv(path, low_memory=False) if path.exists() else pd.DataFrame()

nodal_only_stacks_raw = read_optional_csv(NODAL_ONLY_FILES['stacks'])
nodal_only_members_raw = read_optional_csv(NODAL_ONLY_FILES['members'])
nodal_only_files_raw = read_optional_csv(NODAL_ONLY_FILES['files'])
nodal_only_receivers_raw = read_optional_csv(NODAL_ONLY_FILES['receivers'])
nodal_only_errors_raw = read_optional_csv(NODAL_ONLY_FILES['errors'])

print('Nodal-only stacks:', len(nodal_only_stacks_raw))
print('Nodal-only members:', len(nodal_only_members_raw))
print('Nodal-only indexed files:', len(nodal_only_files_raw))
print('Nodal-only receiver contributions:', len(nodal_only_receivers_raw))
print('Nodal-only processing errors:', len(nodal_only_errors_raw))


Nodal-only stacks: 44
Nodal-only members: 832
Nodal-only indexed files: 396
Nodal-only receiver contributions: 4464
Nodal-only processing errors: 0


## 5. Standardize the stack catalogs

The unified stack table contains one row per completed stacked gather group. It does not merge waveform products.

Source-position authority is retained explicitly:

- `surveyed_geode_source`
- `assigned_nodal_only_source`


In [5]:
def standardize_geode_stacks(frame):
    if frame.empty:
        return pd.DataFrame()

    frame = numeric(
        frame,
        [
            'file_no', 'source_x_truth_m',
            'n_candidate_members', 'n_accepted_members',
            'n_rejected_members', 'median_xcorr_shift_s',
            'median_xcorr_corrcoef',
        ],
    )

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'geode_linked'
    out['stack_basis'] = 'nodal_events_linked_to_geode_stack'
    out['line'] = first_present(frame, ['line'])
    out['survey'] = first_present(frame, ['geode_survey', 'survey'])
    out['geode_event_id'] = first_present(frame, ['geode_event_id'])
    out['geode_file_no'] = first_present(frame, ['file_no'])
    out['nodal_only_group_key'] = None
    out['source_x_m'] = first_present(frame, ['source_x_truth_m'])
    out['source_position_authority'] = 'surveyed_geode_source'
    out['source_position_status'] = 'truth_position_from_geode_metadata'
    out['source_type'] = first_present(frame, ['source_type'])
    out['n_candidate_members'] = first_present(frame, ['n_candidate_members'])
    out['n_accepted_members'] = first_present(frame, ['n_accepted_members'])
    out['n_rejected_members'] = first_present(frame, ['n_rejected_members'])
    out['reference_nodal_event_id'] = first_present(
        frame, ['reference_nodal_event_id']
    )
    out['median_xcorr_shift_s'] = first_present(
        frame, ['median_xcorr_shift_s']
    )
    out['median_xcorr_corrcoef'] = first_present(
        frame, ['median_xcorr_corrcoef']
    )
    out['minimum_xcorr_corrcoef'] = np.nan
    out['expected_blows'] = np.nan
    out['components_written'] = np.nan
    out['output_directory'] = first_present(
        frame, ['output_dir', 'output_directory']
    )
    out['status'] = first_present(frame, ['status'], default='ok')
    return out


def standardize_nodal_only_stacks(frame):
    if frame.empty:
        return pd.DataFrame()

    frame = numeric(
        frame,
        [
            'assigned_source_x_m', 'n_accepted_members',
            'expected_blows', 'median_xcorr_shift_s',
            'median_xcorr_corrcoef', 'minimum_xcorr_corrcoef',
        ],
    )

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'nodal_only'
    out['stack_basis'] = first_present(
        frame, ['stack_basis'], default='nodal_only_inferred_group'
    )
    out['line'] = first_present(frame, ['line'], default='T1')
    out['survey'] = first_present(
        frame, ['survey', 'nodal_timewindow_label']
    )
    out['geode_event_id'] = None
    out['geode_file_no'] = np.nan
    out['nodal_only_group_key'] = first_present(frame, ['group_key'])
    out['source_x_m'] = first_present(frame, ['assigned_source_x_m'])
    out['source_position_authority'] = 'assigned_nodal_only_source'
    out['source_position_status'] = first_present(
        frame,
        ['source_position_status'],
        default='assigned_in_94b_reviewed_in_94c',
    )
    out['source_type'] = first_present(frame, ['source_type'])
    out['n_candidate_members'] = first_present(
        frame, ['n_candidate_members', 'n_accepted_members']
    )
    out['n_accepted_members'] = first_present(
        frame, ['n_accepted_members']
    )
    out['n_rejected_members'] = first_present(
        frame, ['n_rejected_members'], default=0
    )
    out['reference_nodal_event_id'] = first_present(
        frame, ['reference_nodal_event_id']
    )
    out['median_xcorr_shift_s'] = first_present(
        frame, ['median_xcorr_shift_s']
    )
    out['median_xcorr_corrcoef'] = first_present(
        frame, ['median_xcorr_corrcoef']
    )
    out['minimum_xcorr_corrcoef'] = first_present(
        frame, ['minimum_xcorr_corrcoef']
    )
    out['expected_blows'] = first_present(frame, ['expected_blows'])
    out['components_written'] = first_present(
        frame, ['components_written']
    )
    out['output_directory'] = first_present(
        frame, ['output_directory', 'output_dir']
    )
    out['status'] = first_present(
        frame, ['status'], default='stack_created'
    )
    return out


geode_stacks = standardize_geode_stacks(geode_stacks_raw)
nodal_only_stacks = standardize_nodal_only_stacks(nodal_only_stacks_raw)

unified_stacks = pd.concat(
    [geode_stacks, nodal_only_stacks],
    ignore_index=True,
    sort=False,
)

unified_stacks['source_x_m'] = pd.to_numeric(
    unified_stacks['source_x_m'], errors='coerce'
)

if TARGET_LINES is not None:
    unified_stacks = unified_stacks.loc[
        unified_stacks.line.astype(str).isin(TARGET_LINES)
    ].copy()

duplicate_stack_ids = unified_stacks.loc[
    unified_stacks.stack_id.duplicated(keep=False),
    ['stack_id', 'catalog_branch'],
]

if len(duplicate_stack_ids):
    display(duplicate_stack_ids)
    raise RuntimeError('Duplicate stack_id values occur in the unified catalog.')

unified_stacks = cluster_positions(
    unified_stacks,
    tolerance_m=SHOT_POSITION_TOLERANCE_M,
)

unified_stacks = unified_stacks.sort_values(
    ['line', 'canonical_source_x_m', 'catalog_branch', 'stack_id'],
    na_position='last',
).reset_index(drop=True)

print('Unified stacks:', len(unified_stacks))
print('Position clusters:', unified_stacks.source_cluster_id.nunique())
display(unified_stacks.head(20))


Unified stacks: 45
Position clusters: 42


,stack_id,catalog_branch,stack_basis,line,survey,geode_event_id,geode_file_no,nodal_only_group_key,source_x_m,source_position_authority,source_position_status,source_type,n_candidate_members,n_accepted_members,n_rejected_members,reference_nodal_event_id,median_xcorr_shift_s,median_xcorr_corrcoef,minimum_xcorr_corrcoef,expected_blows,components_written,output_directory,status,source_cluster_number,canonical_source_x_m,cluster_min_x_m,cluster_max_x_m,cluster_span_m,cluster_n_stacks,cluster_contains_geode_linked,cluster_contains_nodal_only,source_cluster_id
0,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY19_010M,10.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,43,43,0,T1_N3_Nodal3_T1_N3_E00007,-0.004,0.968000,0.8652,60.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,1.0,10.0,10.0,10.0,0.0,1,False,True,T1_SRC_0001
1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY19_036M,36.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,22,22,0,RECOV_MAY19_036M_0005,-0.002,0.978900,0.9356,22.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,2.0,36.0,36.0,36.0,0.0,1,False,True,T1_SRC_0002
2,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,geode_linked,nodal_events_linked_to_geode_stack,T1,T1_1m_refraction,GEODE_T1_1M_REFRACTION_F3006,3006.0,None,84.5,surveyed_geode_source,truth_position_from_geode_metadata,hammer,6,6,0,T1_N2_Refraction1m_T1_N2_E00007,-0.036,0.945346,NaN,NaN,NaN,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok,3.0,84.5,84.5,84.5,0.0,1,True,False,T1_SRC_0003
3,NODALONLYSTACK_MAY17_104M_x0104.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_104M,104.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,7,7,0,T1_N2_Nodal1_T1_N2_E00545,-0.054,0.953000,0.6816,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,4.0,104.0,104.0,104.0,0.0,1,False,True,T1_SRC_0004
4,NODALONLYSTACK_MAY17_105M_x0105.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_105M,105.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,20,20,0,T1_N2_Nodal1_T1_N2_E00558,-0.025,0.977200,0.9356,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,5.0,105.0,105.0,105.0,0.0,1,False,True,T1_SRC_0005
5,NODALONLYSTACK_MAY17_106M_x0106.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_106M,106.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,27,27,0,T1_N2_Nodal1_T1_N2_E00583,-0.050,0.746500,0.6544,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,6.0,106.0,106.0,106.0,0.0,1,False,True,T1_SRC_0006
6,NODALONLYSTACK_MAY17_107M_x0107.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_107M,107.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,21,21,0,T1_N2_Nodal1_T1_N2_E00617,-0.024,0.716400,0.6544,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,7.0,107.0,107.0,107.0,0.0,1,False,True,T1_SRC_0007
7,NODALONLYSTACK_MAY17_108M_x0108.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_108M,108.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,19,19,0,T1_N2_Nodal1_T1_N2_E00642,0.004,0.954800,0.8852,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,8.0,108.0,108.0,108.0,0.0,1,False,True,T1_SRC_0008
8,NODALONLYSTACK_MAY17_109M_x0109.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_109M,109.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,21,21,0,T1_N2_Nodal1_T1_N2_E00669,0.002,0.959300,0.8669,20.0,"Z,N,E",/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,stack_created,9.0,109.0,109.0,109.0,0.0,1,False,True,T1_SRC_0009
9,NODALONLYSTACK_MAY17_110M_x0110.0m,nodal_only,nodal_only_inferred_group,T1,NaN,None,NaN,MAY17_110M,110.0,assigned_nodal_only_source,assigned_in_94b_reviewed_in_94c,NaN,26,26,0,T1_N2_Nodal1_T1_N2_E00694,0.004,0.928300,0.6

## 6. Standardize indexed output files

In [6]:
def standardize_files(frame, branch):
    if frame.empty:
        return pd.DataFrame(columns=[
            'stack_id', 'catalog_branch', 'component', 'file_type',
            'file_path', 'n_traces', 'n_stack_members',
        ])

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = branch
    out['component'] = first_present(frame, ['component']).map(
        normalize_component
    )
    out['file_type'] = first_present(frame, ['file_type']).astype(str)
    out['file_path'] = first_present(frame, ['file_path']).astype(str)
    out['n_traces'] = pd.to_numeric(
        first_present(frame, ['n_traces']), errors='coerce'
    )
    out['n_stack_members'] = pd.to_numeric(
        first_present(frame, ['n_stack_members']), errors='coerce'
    )
    return out


unified_files = pd.concat(
    [
        standardize_files(geode_files_raw, 'geode_linked'),
        standardize_files(nodal_only_files_raw, 'nodal_only'),
    ],
    ignore_index=True,
    sort=False,
)

unified_files['file_exists'] = unified_files.file_path.map(existing_path)
unified_files['file_size_bytes'] = unified_files.file_path.map(
    lambda value: (
        Path(str(value)).stat().st_size
        if existing_path(value) else np.nan
    )
)

unified_files = unified_files.merge(
    unified_stacks[
        [
            'stack_id', 'line', 'source_x_m', 'canonical_source_x_m',
            'source_cluster_id', 'source_position_authority',
        ]
    ],
    on='stack_id',
    how='inner',
    validate='many_to_one',
)

print('Unified indexed files:', len(unified_files))
print('Missing indexed files:', int((~unified_files.file_exists).sum()))
display(
    unified_files.groupby(
        ['catalog_branch', 'component', 'file_type', 'file_exists'],
        dropna=False,
    ).size().reset_index(name='n_files')
)


Unified indexed files: 399
Missing indexed files: 0


,catalog_branch,component,file_type,file_exists,n_files
0,geode_linked,Z,mseed,True,1
1,geode_linked,Z,png_wiggle,True,1
2,geode_linked,Z,segy,True,1
3,nodal_only,E,mseed,True,44
4,nodal_only,E,png_wiggle,True,44
5,nodal_only,E,segy,True,44
6,nodal_only,N,mseed,True,44
7,nodal_only,N,png_wiggle,True,44
8,nodal_only,N,segy,True,44
9,nodal_only,Z,mseed,True,44


## 7. Standardize event-member provenance

In [7]:
def standardize_geode_members(frame):
    if frame.empty:
        return pd.DataFrame()

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'geode_linked'
    out['nodal_event_id'] = first_present(frame, ['nodal_event_id'])
    out['event_time_utc'] = first_present(
        frame, ['nodal_event_time_utc', 'event_time_utc']
    )
    out['corrected_event_time_utc'] = first_present(
        frame,
        ['corrected_nodal_event_time_utc', 'corrected_event_time_utc'],
    )
    out['xcorr_shift_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_s']), errors='coerce'
    )
    out['xcorr_corrcoef'] = pd.to_numeric(
        first_present(frame, ['xcorr_corrcoef']), errors='coerce'
    )
    out['xcorr_n_traces'] = pd.to_numeric(
        first_present(frame, ['xcorr_n_traces']), errors='coerce'
    )
    out['xcorr_shift_mad_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_mad_s']), errors='coerce'
    )
    out['is_reference_event'] = as_bool(
        first_present(frame, ['is_reference_event'])
    )
    out['included_in_stack'] = as_bool(
        first_present(
            frame,
            ['included_in_output_stack', 'accepted_for_stack'],
        )
    )
    out['waveform_qc_status'] = first_present(
        frame, ['waveform_qc_status']
    )
    out['source_member_path'] = first_present(
        frame, ['long_mseed_path', 'mseed_path']
    )
    return out


def standardize_nodal_only_members(frame):
    if frame.empty:
        return pd.DataFrame()

    out = pd.DataFrame(index=frame.index)
    out['stack_id'] = frame['stack_id'].astype(str)
    out['catalog_branch'] = 'nodal_only'
    out['nodal_event_id'] = first_present(frame, ['nodal_event_id'])
    out['event_time_utc'] = first_present(
        frame, ['event_time_utc', 'event_time']
    )
    out['corrected_event_time_utc'] = first_present(
        frame, ['corrected_event_time_utc']
    )
    out['xcorr_shift_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_s']), errors='coerce'
    )
    out['xcorr_corrcoef'] = pd.to_numeric(
        first_present(frame, ['xcorr_corrcoef']), errors='coerce'
    )
    out['xcorr_n_traces'] = pd.to_numeric(
        first_present(frame, ['xcorr_n_traces']), errors='coerce'
    )
    out['xcorr_shift_mad_s'] = pd.to_numeric(
        first_present(frame, ['xcorr_shift_mad_s']), errors='coerce'
    )
    out['is_reference_event'] = as_bool(
        first_present(frame, ['is_reference_event'])
    )
    out['included_in_stack'] = as_bool(
        first_present(frame, ['included_in_stack'], default=True)
    )
    out['waveform_qc_status'] = first_present(
        frame, ['waveform_qc_status']
    )
    out['source_member_path'] = first_present(
        frame, ['resolved_gather_path']
    )
    return out


unified_members = pd.concat(
    [
        standardize_geode_members(geode_members_raw),
        standardize_nodal_only_members(nodal_only_members_raw),
    ],
    ignore_index=True,
    sort=False,
)

if len(unified_members):
    unified_members = unified_members.merge(
        unified_stacks[
            [
                'stack_id', 'line', 'source_x_m',
                'canonical_source_x_m', 'source_cluster_id',
            ]
        ],
        on='stack_id',
        how='inner',
        validate='many_to_one',
    )

print('Unified accepted/rejected member records:', len(unified_members))
if len(unified_members):
    display(
        unified_members.groupby(
            ['catalog_branch', 'included_in_stack'],
            dropna=False,
        ).size().reset_index(name='n_members')
    )


Unified accepted/rejected member records: 838


,catalog_branch,included_in_stack,n_members
0,geode_linked,True,6
1,nodal_only,True,832


## 8. Build a receiver-level catalog

For the nodal-only branch, receiver provenance comes directly from
`nodal_only_stack_trace_contributions.csv`.

For the Geode-linked branch, receiver positions are read from each indexed
MiniSEED stack when available. Position-coded station names are used only as a
fallback.


In [8]:
try:
    from obspy import read
except Exception as exc:
    read = None
    print('WARNING: ObsPy unavailable; Geode-linked receiver catalog will be limited.')
    print(repr(exc))


def station_code_x_m(station):
    try:
        return int(str(station)) / 100.0
    except Exception:
        return np.nan


receiver_rows = []

# Direct nodal-only contribution catalog.
if len(nodal_only_receivers_raw):
    for row in nodal_only_receivers_raw.itertuples(index=False):
        receiver_rows.append({
            'stack_id': str(row.stack_id),
            'catalog_branch': 'nodal_only',
            'component': normalize_component(getattr(row, 'component', '')),
            'station': str(getattr(row, 'station', '')),
            'channel': str(getattr(row, 'channel', '')),
            'receiver_x_m': pd.to_numeric(
                getattr(row, 'receiver_x_m', np.nan),
                errors='coerce',
            ),
            'n_contributing_events': pd.to_numeric(
                getattr(row, 'n_contributing_events', np.nan),
                errors='coerce',
            ),
            'receiver_position_source': 'nodal_only_trace_contribution_catalog',
            'source_file_path': None,
        })

# Read one MiniSEED file per Geode-linked stack/component.
geode_mseed = unified_files.loc[
    unified_files.catalog_branch.eq('geode_linked')
    & unified_files.file_type.str.lower().eq('mseed')
    & unified_files.file_exists
].copy()

if read is not None:
    for file_row in geode_mseed.itertuples(index=False):
        try:
            stream = read(str(file_row.file_path))
            for trace in stream:
                receiver_x = pd.to_numeric(
                    getattr(trace.stats, 'receiver_x_m', np.nan),
                    errors='coerce',
                )
                if pd.isna(receiver_x):
                    receiver_x = station_code_x_m(trace.stats.station)

                receiver_rows.append({
                    'stack_id': str(file_row.stack_id),
                    'catalog_branch': 'geode_linked',
                    'component': normalize_component(trace.stats.channel),
                    'station': str(trace.stats.station),
                    'channel': str(trace.stats.channel),
                    'receiver_x_m': float(receiver_x),
                    'n_contributing_events': file_row.n_stack_members,
                    'receiver_position_source': (
                        'mseed_trace_stats_or_position_station_code'
                    ),
                    'source_file_path': str(file_row.file_path),
                })
        except Exception as exc:
            print(
                'WARNING: could not inspect receiver metadata:',
                file_row.file_path, repr(exc)
            )

unified_receivers = pd.DataFrame(receiver_rows)

if len(unified_receivers):
    unified_receivers['receiver_x_m'] = pd.to_numeric(
        unified_receivers.receiver_x_m, errors='coerce'
    )
    unified_receivers = unified_receivers.merge(
        unified_stacks[
            [
                'stack_id', 'line', 'source_x_m',
                'canonical_source_x_m', 'source_cluster_id',
            ]
        ],
        on='stack_id',
        how='inner',
        validate='many_to_one',
    )
    unified_receivers = unified_receivers.sort_values(
        ['line', 'canonical_source_x_m', 'stack_id', 'component', 'receiver_x_m']
    ).reset_index(drop=True)

print('Unified receiver records:', len(unified_receivers))
if len(unified_receivers):
    display(
        unified_receivers.groupby(
            ['catalog_branch', 'component'],
            dropna=False,
        ).agg(
            n_receiver_records=('receiver_x_m', 'size'),
            n_unique_positions=('receiver_x_m', 'nunique'),
            receiver_x_min_m=('receiver_x_m', 'min'),
            receiver_x_max_m=('receiver_x_m', 'max'),
        ).reset_index()
    )


Unified receiver records: 4498


,catalog_branch,component,n_receiver_records,n_unique_positions,receiver_x_min_m,receiver_x_max_m
0,geode_linked,Z,34,34,68.09,184.05
1,nodal_only,E,1488,51,28.00,216.00
2,nodal_only,N,1488,51,28.00,216.00
3,nodal_only,Z,1488,51,28.00,216.00


## 9. Standardize processing errors

In [9]:
def standardize_errors(frame, branch):
    if frame.empty:
        return pd.DataFrame(columns=[
            'catalog_branch', 'stack_id', 'group_or_event_id',
            'stage', 'error', 'traceback',
        ])

    out = pd.DataFrame(index=frame.index)
    out['catalog_branch'] = branch
    out['stack_id'] = first_present(frame, ['stack_id'])
    out['group_or_event_id'] = first_present(
        frame, ['group_key', 'geode_event_id']
    )
    out['stage'] = first_present(frame, ['stage'])
    out['error'] = first_present(frame, ['error'])
    out['traceback'] = first_present(frame, ['traceback'])
    return out


unified_errors = pd.concat(
    [
        standardize_errors(geode_errors_raw, 'geode_linked'),
        standardize_errors(nodal_only_errors_raw, 'nodal_only'),
    ],
    ignore_index=True,
    sort=False,
)

print('Unified processing errors:', len(unified_errors))
if len(unified_errors):
    display(
        unified_errors.groupby(
            ['catalog_branch', 'stage'], dropna=False
        ).size().reset_index(name='n_errors')
    )


Unified processing errors: 193


,catalog_branch,stage,n_errors
0,geode_linked,process_stack,193


## 10. Candidate cross-branch source matches

These rows identify proximity clusters containing both branches. They are intended
for review before any waveform combination.

A cluster can contain:

- one Geode-linked stack and one nodal-only stack;
- multiple records from either branch;
- records that are spatially close but belong to different acquisition sequences.

Therefore `candidate_same_source_position=True` means only that the source
coordinates fall within the current clustering tolerance.


In [10]:
cross_branch_clusters = unified_stacks.loc[
    unified_stacks.cluster_contains_geode_linked
    & unified_stacks.cluster_contains_nodal_only
].copy()

candidate_matches = []

for cluster_id, group in cross_branch_clusters.groupby('source_cluster_id'):
    geode_group = group.loc[group.catalog_branch.eq('geode_linked')]
    nodal_only_group = group.loc[group.catalog_branch.eq('nodal_only')]

    for geode_row in geode_group.itertuples(index=False):
        for nodal_row in nodal_only_group.itertuples(index=False):
            distance = abs(
                float(geode_row.source_x_m)
                - float(nodal_row.source_x_m)
            )
            candidate_matches.append({
                'source_cluster_id': cluster_id,
                'line': geode_row.line,
                'canonical_source_x_m': geode_row.canonical_source_x_m,
                'geode_linked_stack_id': geode_row.stack_id,
                'geode_source_x_m': geode_row.source_x_m,
                'geode_survey': geode_row.survey,
                'geode_file_no': geode_row.geode_file_no,
                'nodal_only_stack_id': nodal_row.stack_id,
                'nodal_only_source_x_m': nodal_row.source_x_m,
                'nodal_only_group_key': nodal_row.nodal_only_group_key,
                'source_distance_m': distance,
                'candidate_same_source_position': (
                    distance <= SHOT_POSITION_TOLERANCE_M
                ),
                'waveform_review_status': 'not_reviewed',
                'recommended_action': (
                    'review acquisition timing and waveform correlation'
                ),
            })

candidate_cross_branch_matches = pd.DataFrame(candidate_matches)

print('Cross-branch proximity clusters:', cross_branch_clusters.source_cluster_id.nunique())
print('Candidate cross-branch stack pairs:', len(candidate_cross_branch_matches))
if len(candidate_cross_branch_matches):
    display(
        candidate_cross_branch_matches.sort_values(
            ['line', 'canonical_source_x_m', 'source_distance_m']
        )
    )


Cross-branch proximity clusters: 0
Candidate cross-branch stack pairs: 0


## 11. Integrity checks

In [11]:
issues = []

if unified_stacks.empty:
    issues.append('No stack rows were loaded.')

if unified_stacks.source_x_m.isna().any():
    issues.append(
        f'{int(unified_stacks.source_x_m.isna().sum())} stacks have no source position.'
    )

missing_stack_files = unified_files.loc[~unified_files.file_exists].copy()
if len(missing_stack_files):
    issues.append(
        f'{len(missing_stack_files)} indexed output files are missing.'
    )

stacks_without_mseed = sorted(
    set(unified_stacks.stack_id)
    - set(
        unified_files.loc[
            unified_files.file_type.str.lower().eq('mseed')
            & unified_files.file_exists,
            'stack_id',
        ]
    )
)
if stacks_without_mseed:
    issues.append(
        f'{len(stacks_without_mseed)} stacks have no existing indexed MiniSEED file.'
    )

clusters_too_wide = unified_stacks.loc[
    unified_stacks.cluster_span_m > SHOT_POSITION_TOLERANCE_M + 1e-9
]
if len(clusters_too_wide):
    issues.append(
        f'{clusters_too_wide.source_cluster_id.nunique()} proximity clusters '
        'span more than the configured tolerance.'
    )

print('Integrity issues:', len(issues))
for issue in issues:
    print(' -', issue)

if len(missing_stack_files):
    display(
        missing_stack_files[
            ['stack_id', 'catalog_branch', 'component', 'file_type', 'file_path']
        ].head(50)
    )

if stacks_without_mseed:
    display(
        unified_stacks.loc[
            unified_stacks.stack_id.isin(stacks_without_mseed),
            ['stack_id', 'catalog_branch', 'line', 'source_x_m', 'status'],
        ]
    )

if not issues:
    print('PASS: both nodal stack branches were cataloged and all indexed MiniSEED files exist.')
else:
    print('REVIEW: the catalog will still be exported, but inspect the issues above.')


Integrity issues: 0
PASS: both nodal stack branches were cataloged and all indexed MiniSEED files exist.


## 12. Export the unified catalog

In [12]:
OUTPUTS = {
    'stacks': OUT_ROOT / '96_unified_nodal_stacks.csv',
    'files': OUT_ROOT / '96_unified_nodal_stack_files.csv',
    'members': OUT_ROOT / '96_unified_nodal_stack_members.csv',
    'receivers': OUT_ROOT / '96_unified_nodal_stack_receivers.csv',
    'errors': OUT_ROOT / '96_unified_nodal_stack_processing_errors.csv',
    'candidate_matches': OUT_ROOT / '96_candidate_cross_branch_source_matches.csv',
    'summary': OUT_ROOT / '96_unified_nodal_catalog_summary.csv',
}

unified_stacks.to_csv(OUTPUTS['stacks'], index=False)
unified_files.to_csv(OUTPUTS['files'], index=False)
unified_members.to_csv(OUTPUTS['members'], index=False)
unified_receivers.to_csv(OUTPUTS['receivers'], index=False)
unified_errors.to_csv(OUTPUTS['errors'], index=False)
candidate_cross_branch_matches.to_csv(
    OUTPUTS['candidate_matches'], index=False
)

summary_rows = [
    ('unified_stacks', len(unified_stacks)),
    ('geode_linked_stacks', int(unified_stacks.catalog_branch.eq('geode_linked').sum())),
    ('nodal_only_stacks', int(unified_stacks.catalog_branch.eq('nodal_only').sum())),
    ('source_position_clusters', unified_stacks.source_cluster_id.nunique()),
    ('cross_branch_clusters', cross_branch_clusters.source_cluster_id.nunique()),
    ('candidate_cross_branch_pairs', len(candidate_cross_branch_matches)),
    ('indexed_files', len(unified_files)),
    ('missing_indexed_files', int((~unified_files.file_exists).sum())),
    ('member_records', len(unified_members)),
    ('receiver_records', len(unified_receivers)),
    ('processing_errors', len(unified_errors)),
    ('integrity_issue_count', len(issues)),
]
summary = pd.DataFrame(summary_rows, columns=['metric', 'value'])
summary.to_csv(OUTPUTS['summary'], index=False)

display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:18s} {path}')


,metric,value
0,unified_stacks,45
1,geode_linked_stacks,1
2,nodal_only_stacks,44
3,source_position_clusters,42
4,cross_branch_clusters,0
5,candidate_cross_branch_pairs,0
6,indexed_files,399
7,missing_indexed_files,0
8,member_records,838
9,receiver_records,4498



Written:
  stacks             /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_stacks.csv
  files              /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_stack_files.csv
  members            /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_stack_members.csv
  receivers          /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_stack_receivers.csv
  errors             /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_stack_processing_errors.csv
  candidate_matches  /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_candidate_cross_branch_source_matches.csv
  summary            /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_catalog/96_unified_nodal_catalog_summary.csv


## 13. Recommended next step

Use `96_candidate_cross_branch_source_matches.csv` as the starting review queue.

For each candidate pair:

1. verify that acquisition timing and field context support a repeated source position;
2. compare common receiver traces;
3. estimate gather-level and trace-level correlation;
4. accept or reject the pair for later waveform integration;
5. retain both original stack products and provenance regardless of the decision.

The later T1 real-data integration notebook should consume the unified catalogs rather
than scanning the two upstream output trees independently.
